# Concept Learning in Machine Learning - Starter Notebook
Concept learning is the task of inferring a boolean-valued function from training examples described by attribute-value pairs.

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# Training examples: Sky, AirTemp, Humidity, Wind, Water, Forecast -> EnjoySport?
columns = ['Sky', 'AirTemp', 'Humidity', 'Wind', 'Water', 'Forecast', 'EnjoySport']
data = [
    ['Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same',    'Yes'],
    ['Sunny', 'Warm', 'High',   'Strong', 'Warm', 'Same',    'Yes'],
    ['Rainy', 'Cold', 'High',   'Strong', 'Warm', 'Change',  'No'],
    ['Sunny', 'Warm', 'High',   'Strong', 'Cool', 'Change',  'Yes'],
]
df = pd.DataFrame(data, columns=columns)
print(df.to_string(index=False))

In [ ]:
# Hypothesis representation
# '?' = any value acceptable, '0' = no value acceptable (null), specific = exact match
# Most general hypothesis: ['?', '?', '?', '?', '?', '?']
# Most specific hypothesis: ['0', '0', '0', '0', '0', '0']

FEATURES = columns[:-1]
positive = df[df['EnjoySport'] == 'Yes'][FEATURES].values
negative = df[df['EnjoySport'] == 'No'][FEATURES].values

print(f'Positive examples: {len(positive)}')
print(f'Negative examples: {len(negative)}')

In [ ]:
# Consistent hypothesis check
def is_consistent(h, x):
    """Returns True if hypothesis h is consistent with instance x."""
    return all(hi == '?' or hi == xi for hi, xi in zip(h, x))

# The most specific hypothesis consistent with all positives (Find-S result)
h_specific = list(positive[0])
for x in positive[1:]:
    for i in range(len(h_specific)):
        if h_specific[i] != x[i]:
            h_specific[i] = '?'

print('Most specific consistent hypothesis (Find-S):', h_specific)

# Test on all examples
print('\nConsistency checks:')
for _, row in df.iterrows():
    x = list(row[FEATURES])
    label = row['EnjoySport']
    pred = 'Yes' if is_consistent(h_specific, x) else 'No'
    status = 'OK' if pred == label else 'MISMATCH'
    print(f'  {x} -> predicted={pred}, actual={label} [{status}]')

In [ ]:
# Hypothesis space size illustration
# For d attributes each with n_i values: |H| = product(n_i + 2) + 1
attr_values = {col: df[col].nunique() for col in FEATURES}
H_size = 1
for n in attr_values.values():
    H_size *= (n + 2)
H_size += 1  # add the null hypothesis
print('Attribute value counts:', attr_values)
print(f'Syntactically distinct hypotheses: {H_size}')